# Notebook 01 — Stack Biestacional Sentinel-1 + Sentinel-2 (100% Cómputo en SEPAL)

- **Instancia SEPAL recomendada:** **`m8d`** (8 CPUs, 32 GB RAM, 474 GB SSD en `/tmp`, 0.53 USD/h) o **`m16d`** (16 CPUs, 64 GB RAM, 950 GB SSD, 1.06 USD/h).
- **Cero descargas lentas de GEE/Drive:**
  - **Sentinel-2 Collection 1 (`sentinel-2-c1-l2a`):** leído directo por ventanas COG desde **AWS Earth Search STAC** (sin credenciales), con verificación física de offset radiométrico y protección `MNDWI + SCL` para manglares inundados.
  - **Sentinel-1 ARD (`OPERA_RTC_S1` o `GRD_SNAP`):** descargado desde **NASA ASF** y filtrado con **Refined Lee 7×7 en potencia lineal** vectorizado en Python multi-núcleo sobre SEPAL.
  - **Salida:** `outputs/stack/stack_biestacional_<region>_2025_v1_sepal.tif` (**24 bandas float32**) y `qa_stack_...tif` (**6 bandas uint16**) comprimidos en tu disco persistente de SEPAL.

In [ ]:
# 1. Configuracion de corrida — elegir REGION: "CARIBE" o "PACIFICO"
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config_utils import ensure_output_dirs, load_config, resolve_max_workers
from src.local_aoi import compute_region_grid, ensure_local_aoi
from src.local_s1 import get_earthdata_session, has_earthdata_credentials, inspect_s1_orbits_asf
from src.local_s2 import search_s2_stac
from src.local_stack import (
    add_local_raster_overlay,
    build_local_biseasonal_stack,
    diagnose_local_stack,
    get_local_stack_paths,
)

# >>> CAMBIAR AQUI ENTRE "CARIBE" Y "PACIFICO" (recomendado probar primero CARIBE) <<<
REGION = "PACIFICO"

cfg = load_config()
dirs = ensure_output_dirs(cfg)
reg_cfg = cfg["regiones"][REGION]

aoi_gdf = ensure_local_aoi(REGION, cfg)
grid_info = compute_region_grid(
    aoi_gdf,
    crs=reg_cfg["crs"],
    scale_m=float(cfg["stack"]["escala_m"]),
    buffer_m=float(cfg["stack"]["buffer_stack_m"]),
    margen_kernel_m=float(cfg["stack"]["margen_kernel_m"]),
)

print(f"Region          : {REGION} ({reg_cfg['nombre_largo']})")
print(f"CRS / Escala    : {grid_info['crs']} @ {grid_info['scale_m']} m")
print(f"Grilla exacta   : {grid_info['height']} filas x {grid_info['width']} cols ({grid_info['height']*grid_info['width']/1e6:.2f} M px)")
print(f"Nucleos SEPAL   : {resolve_max_workers(cfg)} workers en paralelo")
print(f"Fuente S1 local : {cfg['s1']['fuente_local']} | Speckle: {cfg['s1']['speckle_filter']}")
print(f"NASA Earthdata  : {'[OK] Listo' if has_earthdata_credentials() else '[FALTA] Configura ~/.netrc en Notebook 00'}")

In [ ]:
# 2. Inventario rapido (sin descargar pixeles): escenas S2 en STAC y orbitas S1 en NASA ASF
fuente_s1 = cfg["s1"]["fuente_local"]
filas_s1 = []
filas_s2 = []

for temp in ["SECA", "LLUVIA"]:
    t_cfg = reg_cfg["temporadas"][temp]
    s2_hits = search_s2_stac(grid_info["bounds_4326"], t_cfg["inicio"], t_cfg["fin"], cfg["s2"])
    filas_s2.append({
        "region": REGION,
        "temporada": temp,
        "escenas_candidatas_stac": len(s2_hits),
        "tiles_mgrs": ", ".join(sorted({str(it.properties.get("grid:code", "")) for it in s2_hits})),
    })

    orb_info = inspect_s1_orbits_asf(grid_info["wkt_search_4326"], t_cfg["inicio"], t_cfg["fin"], fuente=fuente_s1)
    for orb, d in orb_info.items():
        filas_s1.append({
            "region": REGION,
            "temporada": temp,
            "orbita": orb,
            "n_productos": d["n_productos"],
            "n_fechas_unicas": d["n_fechas_unicas"],
            "orbitas_relativas": ", ".join(str(x) for x in d["orbitas_relativas"]),
        })

print("=== SENTINEL-2 DISPONIBLE EN AWS EARTH SEARCH ===")
display(pd.DataFrame(filas_s2))
print(f"=== SENTINEL-1 DISPONIBLE EN NASA ASF ({fuente_s1}) ===")
display(pd.DataFrame(filas_s1))
print("Orbita configurada en config.yaml:", reg_cfg["orbit_pass"])

In [ ]:
# 3. Construir el Stack Biestacional de 24 bandas + QA 6 bandas en el disco de SEPAL
# Cambia EJECUTAR_PROCESAMIENTO_LOCAL = True una vez aprobadas las orbitas de la celda 2.
EJECUTAR_PROCESAMIENTO_LOCAL = False

paths = get_local_stack_paths(REGION, cfg)
if EJECUTAR_PROCESAMIENTO_LOCAL:
    session = get_earthdata_session()
    bundle = build_local_biseasonal_stack(REGION, cfg, session=session)
    print(f"[OK] Stack guardado en : {bundle['stack_path']}")
    print(f"[OK] QA guardado en    : {bundle['qa_path']}")
    print(f"S2 Seca/Lluvia validas : {bundle['s2_seca']['n_escenas_validas']} / {bundle['s2_lluvia']['n_escenas_validas']}")
    print(f"S1 Seca/Lluvia fechas  : {bundle['s1_seca']['n_fechas_procesadas']} / {bundle['s1_lluvia']['n_fechas_procesadas']}")
else:
    if paths["stack"].is_file():
        print(f"[INFO] Ya existe Stack previo en disco: {paths['stack']}")
    else:
        print("Modo seguro (EJECUTAR_PROCESAMIENTO_LOCAL = False). Cambialo a True para procesar el Stack en SEPAL.")

In [ ]:
# 4. Control de calidad cuantitativo del Stack local (sin GEE)
assert paths["stack"].is_file() and paths["qa"].is_file(), "Ejecuta primero la celda 3 para generar el Stack local."

diag = diagnose_local_stack(paths["stack"], paths["qa"], seed=int(cfg["proyecto"]["seed"]))
print(f"=== DIAGNOSTICO STACK LOCAL {REGION} ===")
print(f"Dimensiones      : {diag['dimensiones']} | CRS: {diag['crs']} | Tamano GeoTIFF: {diag['tamano_stack_mb']:.1f} MB")
print(f"Bandas ({diag['n_bandas_stack']}): {diag['bandas_stack']}")
print(f"Pixeles validos dentro del AOI oficial: {diag['fraccion_valida_aoi']*100:.2f}% ({diag['n_pixeles_validos_aoi']:,} / {diag['n_pixeles_aoi_base']:,})")

display(diag["stats_df"])
diag["stats_df"].to_csv(paths["diagnostico"], index=False)
print(f"Control guardado en: {paths['diagnostico']}")

In [ ]:
# 5. Control visual interactivo en SEPAL (sin localtileserver ni GEE)
import geemap
from src.local_aoi import get_local_guia_path

m = geemap.Map(height="560px")
m.add_basemap("SATELLITE")

b = aoi_gdf.total_bounds
m.setCenter((b[0] + b[2]) / 2.0, (b[1] + b[3]) / 2.0, 10)

add_local_raster_overlay(m, paths["stack"], ["B8_SECA", "B11_SECA", "B4_SECA"], "Falsocolor Manglar (B8-B11-B4 Seca)", stretch=(0.0, 0.40), visible=True)
add_local_raster_overlay(m, paths["stack"], ["B4_SECA", "B3_SECA", "B2_SECA"], "S2 Seca RGB", stretch=(0.0, 0.25), visible=False)
add_local_raster_overlay(m, paths["stack"], ["VH_SECA"], "SAR VH Seca (dB)", stretch=(-28.0, -8.0), palette=["#0015FF", "#00FFFF", "#FFFF00", "#FF0000"], visible=False)

guia_tif = get_local_guia_path(REGION, cfg)
if guia_tif.is_file():
    add_local_raster_overlay(m, guia_tif, [1], "Guia DinMangle 2020-2022", categorical=True, palette={1: "#00FF66"}, opacity=0.55, visible=False)

m.add_gdf(aoi_gdf, layer_name=f"AOI {REGION}", style={"color": "#FFFFFF", "fillOpacity": 0.0, "weight": 2})
m